# Stage 2-3b — EfficientNet-B0 Fine-tuning

**Pipeline**: `preprocess()` → `DataLoader (torchvision transforms)` → `DeepClassifier (EfficientNet-B0)`

**Training strategy — two phases**:
- **Phase 1** (head only, ~10 epochs, lr=1e-3): backbone frozen. Trains only the custom head. Avoids catastrophic forgetting of ImageNet features.
- **Phase 2** (full network, ~10 epochs, lr=1e-5): backbone unfrozen at low lr. Adapts feature extractor to industrial textures.

**Goal**: beat the HOG+SVM baseline (F1=0.605, Recall_defect=46%) with learned features.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch import nn, optim

from src.dataset import MVTecTorchDataset
from src.models.deep import DeepClassifier, get_transforms
from src.evaluate import evaluate_classification, print_results, results_row

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CATEGORY     = 'metal_nut'
DATA_ROOT    = Path('../data/mvtec_ad')
RANDOM_STATE = 42
BATCH_SIZE   = 32
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 10

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Dataset and DataLoaders

Same stratified 70/30 split as HOG+SVM (same `random_state=42`) → results are directly comparable.

In [ ]:
all_paths, all_labels = MVTecTorchDataset.collect_paths(DATA_ROOT / CATEGORY)
all_labels_arr = np.array(all_labels)

train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_paths, all_labels, test_size=0.30,
    random_state=RANDOM_STATE, stratify=all_labels
)

train_ds = MVTecTorchDataset(train_paths, train_labels, transform=get_transforms(train=True))
test_ds  = MVTecTorchDataset(test_paths,  test_labels,  transform=get_transforms(train=False))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds)} samples  (good: {train_labels.count(0)}, defective: {train_labels.count(1)})')
print(f'Test:  {len(test_ds)}  samples  (good: {test_labels.count(0)},  defective: {test_labels.count(1)})')

## 2. Model — EfficientNet-B0 with custom head

In [ ]:
model = DeepClassifier(num_classes=2, backbone='efficientnet_b0', freeze_backbone=True)
model = model.to(DEVICE)
print(model)

params = model.count_parameters()
print(f'\nTrainable params (Phase 1): {params["trainable"]:,}')
print(f'Frozen params:              {params["frozen"]:,}')
print(f'Total params:               {params["total"]:,}')

## 3. Training loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (model(imgs).argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * len(labels)
        correct    += (out.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n

criterion = nn.CrossEntropyLoss()
history   = {'phase': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
print('Training functions ready.')

### Phase 1 — Head only (backbone frozen, lr=1e-3)

In [ ]:
optimizer_p1 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

print('Phase 1 — training head only')
print(f'{"Epoch":>6} {"Train Loss":>12} {"Train Acc":>10} {"Val Loss":>10} {"Val Acc":>8}')
print('-' * 52)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tl, ta = train_epoch(model, train_loader, optimizer_p1, criterion, DEVICE)
    vl, va = eval_epoch(model,  test_loader,  criterion, DEVICE)
    history['phase'].append(1)
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'{epoch:>6} {tl:>12.4f} {ta:>10.4f} {vl:>10.4f} {va:>8.4f}')

### Phase 2 — Full network (backbone unfrozen, lr=1e-5)

In [ ]:
model.unfreeze_backbone()
params = model.count_parameters()
print(f'Backbone unfrozen — trainable params: {params["trainable"]:,}')

optimizer_p2 = optim.Adam(model.parameters(), lr=1e-5)

print('\nPhase 2 — full network fine-tuning')
print(f'{"Epoch":>6} {"Train Loss":>12} {"Train Acc":>10} {"Val Loss":>10} {"Val Acc":>8}')
print('-' * 52)

for epoch in range(1, PHASE2_EPOCHS + 1):
    tl, ta = train_epoch(model, train_loader, optimizer_p2, criterion, DEVICE)
    vl, va = eval_epoch(model,  test_loader,  criterion, DEVICE)
    history['phase'].append(2)
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'{epoch:>6} {tl:>12.4f} {ta:>10.4f} {vl:>10.4f} {va:>8.4f}')

## 4. Training curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
phase_boundary = PHASE1_EPOCHS + 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in [
    (axes[0], 'loss', 'Loss'),
    (axes[1], 'acc',  'Accuracy'),
]:
    ax.plot(epochs, history[f'train_{metric}'], label='Train', color='steelblue')
    ax.plot(epochs, history[f'val_{metric}'],   label='Val',   color='tomato')
    ax.axvline(phase_boundary, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.set_title(f'EfficientNet-B0 — {title}')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Training Curves (Phase 1: head only | Phase 2: full network)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Evaluation and comparison with HOG+SVM baseline

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

metrics_dl = evaluate_classification(
    np.array(all_true), np.array(all_preds),
    model_name='EfficientNet-B0 (metal_nut)'
)
print_results(metrics_dl)

In [ ]:
import pandas as pd

# HOG+SVM baseline results (from Commit 4)
baseline = {
    'Model':              'HOG + SVM',
    'Accuracy':           0.8317,
    'F1 (defect)':        0.6047,
    'Recall (defect)':    0.4643,
    'Precision (defect)': 0.8667,
}

df = pd.DataFrame([baseline, results_row(metrics_dl)])
print('=== Model Comparison ===')
print(df.to_string(index=False))

## 6. Save checkpoint

In [ ]:
import os
os.makedirs('../outputs/checkpoints', exist_ok=True)
torch.save(model.state_dict(), '../outputs/checkpoints/efficientnet_metal_nut.pt')
print('Checkpoint saved to outputs/checkpoints/efficientnet_metal_nut.pt')
print('(excluded from git by .gitignore)')

## Summary — Design Decisions

| Choice | Value | Why |
|---|---|---|
| Backbone | EfficientNet-B0 | 5.3M params vs ResNet50's 25M — less overfitting on small dataset |
| Phase 1 lr | 1e-3 | Standard for training a new head from scratch |
| Phase 2 lr | 1e-5 | Low lr prevents catastrophic forgetting of pretrained features |
| Loss | CrossEntropyLoss | Standard for multi-class classification; works for binary (num_classes=2) |
| Batch size | 32 | Fits in 12GB VRAM with room to spare; stable gradient estimates |
| Split | Stratified 70/30 | Same seed as HOG+SVM for direct comparison |